# Spaceship Titanic — Preprocessing & Feature Engineering

This notebook turns the raw Kaggle data into a clean, model-ready dataset. It
covers dropping non-predictive columns, splitting the `Cabin` column, imputing
missing values, and building the `OnboardSpending` feature.

**Design note — avoiding data leakage:** all imputation statistics (mode, median,
mean) are computed on the **training set only** and then applied to both train and
test. The **encoding and scaling** are *not* done here — they are fit inside the
model pipeline in `03_modeling.ipynb`, *after* the train/validation split, so the
validation fold never influences the fitted transformers.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

## 1. Load the raw data

In [2]:
train = pd.read_csv("../Data/train.csv")
test = pd.read_csv("../Data/test.csv")

print(f"Train: {train.shape} | Test: {test.shape}")

Train: (8693, 14) | Test: (4277, 13)


## 2. Drop non-predictive columns

`PassengerId` and `Name` are identifiers with no predictive value, so we drop them
from both sets.

In [3]:
train = train.drop(columns=["PassengerId", "Name"])
test = test.drop(columns=["PassengerId", "Name"])

## 3. Split the `Cabin` column

`Cabin` follows the pattern `Deck/Number/Side`. We split it into separate columns
and drop `Number`, which the EDA showed to be uninformative (hundreds of unique
values, no clear structure).

In [4]:
for df in (train, test):
    df[["Deck", "Number", "Side"]] = df["Cabin"].str.split("/", expand=True)

train = train.drop(columns=["Cabin", "Number"])
test = test.drop(columns=["Cabin", "Number"])

## 4. Missing-value strategy

Every feature column has missing values. We impute them with the following
strategy, computing all statistics **on the training set**:

| Column(s)                       | Strategy                                            |
|---------------------------------|-----------------------------------------------------|
| `HomePlanet`, `Destination`     | most frequent value (mode)                          |
| `Age`                           | median                                              |
| `CryoSleep`                     | logical rule (zero spending ⇒ asleep), else `False` |
| `VIP`                           | `False` (rare status)                               |
| spending columns                | asleep ⇒ 0, otherwise training mean                 |
| `Deck`, `Side`                  | most frequent value (mode)                          |

We capture the training statistics up front so the exact same values are applied
to the test set.

In [5]:
spending_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# Statistics captured from the TRAINING set only.
homeplanet_mode = train["HomePlanet"].mode()[0]
destination_mode = train["Destination"].mode()[0]
age_median = train["Age"].median()
deck_mode = train["Deck"].mode()[0]
side_mode = train["Side"].mode()[0]

print("HomePlanet mode:", homeplanet_mode)
print("Destination mode:", destination_mode)  # TRAPPIST-1e, not Earth
print("Age median:", age_median)

HomePlanet mode: Earth
Destination mode: TRAPPIST-1e
Age median: 27.0


### HomePlanet & Destination

> **Bug fix:** the original notebook filled `Destination` with the mode of
> `HomePlanet` (`Earth`). Each column is now filled with its **own** mode, so
> `Destination` is correctly filled with `TRAPPIST-1e`.

In [6]:
for df in (train, test):
    df["HomePlanet"] = df["HomePlanet"].fillna(homeplanet_mode)
    df["Destination"] = df["Destination"].fillna(destination_mode)

### Age

In [7]:
for df in (train, test):
    df["Age"] = df["Age"].fillna(age_median)

### CryoSleep

Passengers in cryo-sleep cannot spend money. We use this to impute `CryoSleep`:
if the value is missing **and** every spending column is exactly `0`, the
passenger was asleep (`True`); any remaining missing values default to `False`,
the most frequent class.

In [8]:
for df in (train, test):
    asleep = df["CryoSleep"].isna() & (df[spending_cols] == 0).all(axis=1)
    df.loc[asleep, "CryoSleep"] = True
    df["CryoSleep"] = df["CryoSleep"].fillna(False)

### VIP

VIP status is rare, so missing values are filled with `False`.

In [9]:
for df in (train, test):
    df["VIP"] = df["VIP"].fillna(False)

### Spending columns

For passengers in cryo-sleep, missing spending is set to `0`. For awake
passengers, missing spending is filled with the **training mean** of awake
passengers (computed once, applied to both sets).

In [10]:
# Mean spending of awake passengers in the TRAINING set.
awake_means = train.loc[train["CryoSleep"] == False, spending_cols].mean()

for df in (train, test):
    asleep = df["CryoSleep"] == True
    for col in spending_cols:
        df.loc[asleep & df[col].isna(), col] = 0
        df[col] = df[col].fillna(awake_means[col])

### Deck & Side

In [11]:
for df in (train, test):
    df["Deck"] = df["Deck"].fillna(deck_mode)
    df["Side"] = df["Side"].fillna(side_mode)

## 5. Feature engineering: `OnboardSpending`

`OnboardSpending` is the total spend across the five categories. It is computed
**per row from each set's own columns**.

> **Bug fix:** the original notebook computed the test set's `OnboardSpending` from
> the *train* columns, mixing the two datasets. Each set now uses its own values.

In [12]:
for df in (train, test):
    df["OnboardSpending"] = df[spending_cols].sum(axis=1)

## 6. Final check

Confirm there are no missing values left in either set.

In [13]:
print("Train missing:", int(train.isnull().sum().sum()))
print("Test missing: ", int(test.isnull().sum().sum()))
train.head()

Train missing: 0
Test missing:  0


,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Deck,Side,OnboardSpending
0,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False,B,P,0.0
1,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True,F,S,736.0
2,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False,A,S,10383.0
3,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False,A,S,5176.0
4,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True,F,S,1091.0


## 7. Save the processed data

We save with `index=False` and an explicit `.csv` extension so the modelling
notebook can load the data directly without a stray index column.

> **Cleanup vs. original:** the old version saved without `index=False`, which
> created an `Unnamed: 0` column that had to be dropped again after loading.

In [14]:
train.to_csv("../Data/processed_train.csv", index=False)
test.to_csv("../Data/processed_test.csv", index=False)
print("Saved processed_train.csv and processed_test.csv")

Saved processed_train.csv and processed_test.csv
